In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_tickets.csv")

def assign_priority(text):
    text = text.lower()

    high_keywords = [
        "urgent",
        "cannot login",
        "outage",
        "down",
        "security",
        "breach",
        "data loss",
        "vpn not working",
        "server down",
        "email unavailable"
    ]

    medium_keywords = [
        "slow",
        "error",
        "access",
        "permission",
        "storage",
        "hardware",
        "password",
        "microsoft 365",
        "vpn"
    ]

    if any(keyword in text for keyword in high_keywords):
        return "High"
    elif any(keyword in text for keyword in medium_keywords):
        return "Medium"
    else:
        return "Low"

df["priority"] = df["ticket_text"].apply(assign_priority)

print(df["priority"].value_counts())

df.to_csv("../data/processed/tickets_with_priority.csv", index=False)

priority
Low       29096
Medium    15363
High       3378
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

X = df["ticket_text"]
y = df["priority"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

priority_clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ("model", LogisticRegression(max_iter=1000))
])

priority_clf.fit(X_train, y_train)

priority_pred = priority_clf.predict(X_test)

print(classification_report(y_test, priority_pred))

joblib.dump(priority_clf, "../models/priority_classifier.pkl")

              precision    recall  f1-score   support

        High       0.99      0.65      0.79       676
         Low       0.94      1.00      0.97      5819
      Medium       0.96      0.92      0.94      3073

    accuracy                           0.95      9568
   macro avg       0.96      0.86      0.90      9568
weighted avg       0.95      0.95      0.95      9568



['../models/priority_classifier.pkl']